## **Project 2 - Alex Eyre**
### In vivo CRISPR screens identify the E3 ligase Cop1 as a modulator of macrophage infiltration and cancer immunotherapy target
### (Wang et al., Cell 2021)
=============================================================================

In [1]:
# =============================================================================
# Import libraries.
# =============================================================================

import os
import sys
import logging
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from pathlib import Path

import mageck
#sys.path.insert(0, "/mnt/c/Users/aweyr/OneDrive/Documents/Programs/DrugZ")
#import drugz as dz

In [2]:
# =============================================================================
# Startup R.
# =============================================================================

logging.getLogger("rpy2").setLevel(logging.CRITICAL)
%load_ext rpy2.ipython

In [3]:
#%%R
# =============================================================================
# Load R Packages.
# =============================================================================

#suppressPackageStartupMessages({library("DESeq2") # DESeq2: 1.50.2
#                               library("clusterProfiler") # clusterProfiler: 4.18.4
#                               library("ComplexHeatmap") # ComplexHeatmap: 2.26.1
#                               library("enrichplot") # enrichplot: 1.30.5 
#                               library("org.Mm.eg.db") # org.Mm.eg.db: 1.30.5 
#                               library("CellChat")}) # CellChat

=============================================================================
#### Functions
=============================================================================

In [4]:
def parse_mageck_results(
    df: pd.DataFrame,
    *,
    header: bool = True,
    columns_per_comparison: int = 14,
    reset_index: bool = True,
) -> dict[str, pd.DataFrame]:
    
    """
    Parse one or more MAGeCK comparison results from a results table.

    Parameters
    ----------
    df : pd.DataFrame
        Raw MAGeCK results table read with header=None.

    header : bool
        Whether the table contains an additional first row with comparison
        names.

        If True, row 0 contains comparison names and row 1 contains the
        MAGeCK output headers.

        If False, row 0 contains the MAGeCK output headers.

    columns_per_comparison : int
        Number of columns in each MAGeCK comparison block.

    reset_index : bool
        Whether to reset the index after removing metadata/header rows.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary where keys are comparison names and values are cleaned
        MAGeCK result tables.
    """

    mageck_results = {}

    n_comparisons = df.shape[1] // columns_per_comparison

    for comparison_idx in range(n_comparisons):
        start_idx = (comparison_idx * columns_per_comparison) + comparison_idx
        end_idx = start_idx + columns_per_comparison

        subset = df.iloc[:, list(range(start_idx, end_idx))].copy()
        
        if reset_index:
            subset = subset.reset_index(drop = True)

        if header:
            comparison_name = str(subset.columns[0]).strip() 
            subset.columns = subset.iloc[0]
            subset = subset.iloc[1:]
        else:
            comparison_name = f"comparison_{comparison_idx + 1}"

        mageck_results[comparison_name] = subset

    return mageck_results

=============================================================================
#### In Vivo screens with the MusCK library uncover classic and novel regulators of immune evasion.
=============================================================================

In [5]:
# =============================================================================
# Define project paths.
# =============================================================================
project_dir = Path.cwd().parents[0]
data_dir = project_dir / "data" / "supplemental"

In [6]:
# =============================================================================
# Load sgRNA Library Design.
# =============================================================================
MusCK_library_design = pd.read_csv(data_dir / "Table_S1_MusCK_library_design.txt", sep = "\t")

#### MusCK Description
```
sgID: unique identifiers for each sgRNA
seq: sgRNA sequence
geneID: non-unique gene identifier, each gene contains 5 sgRNAs
```

In [7]:
# =============================================================================
# Parse in published supplemental tables.
# =============================================================================
# Experiment 1 - In Vitro
MusCK_S2_invitro = pd.read_csv(data_dir / "Table_S2_In_vitro_MusCK_library_screening_on_4T1_cells.txt", sep = "\t", header = 0)
MusCK_invitro = parse_mageck_results(MusCK_S2_invitro, header = False)

# Experiment 1 - In Vivo
MusCK_S3_invivo = pd.read_csv(data_dir / "Table_S3_In_vivo_MusCK_library_screening_on_4T1_cells.txt", sep = "\t", header = 0)
MusCK_invivo = parse_mageck_results(MusCK_S3_invivo, header = True)

In [8]:
# =============================================================================
# In Vitro Data.
# =============================================================================
MusCK_invitro['comparison_1'].describe(include = 'all').iloc[np.arange(0,4),:]

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4721,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0,4721.0
unique,4721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,FARSA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# =============================================================================
# In Vivo Data - BALB/c VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK_invivo['BALB/c VS. BALB/c Foxn1nu/nu'].describe(include = 'all').iloc[np.arange(0,4),:]

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776
unique,4776,11,4386,4379,393,4776,11,2387,4533,4525,2,4776,5,2387
top,Trim24,5,0.1429,0.33485,0.999773,1,2,-0.58496,0.56331,0.56392,0.99997,4774,0,-0.58496
freq,1,4034,8,8,2421,1,1534,409,73,73,4772,1,4554,410


In [10]:
# =============================================================================
# In Vivo Data - BALB/c +OVA VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK_invivo['BALB/c +OVA VS. BALB/c Foxn1nu/nu'].describe(include = 'all').iloc[np.arange(0,4),:]

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776,4776
unique,4776,11,4425,4402,363,4776,10,2722,4561,4566,13,4776,2,2721
top,TTK,5,0.13604,0.3226,0.999814,1,2,-0.58496,0.79489,0.79475,0.999933,3966,0,-0.58496
freq,1,4055,7,7,2914,1,1536,476,56,56,4735,1,4775,477


#### MusCK Statistics Description

```
id: geneID
num: number of sgRNAs representing the gene
neg|score: gene-level p-value for negative selection (depletion), smaller values indicate stronger evidence that disruption of this gene decreases cell fitness under the tested condition
neg|p-value: one-sided p-value testing whether sgRNAs targeting the gene are significantly depleted relative to the background distribution
neg|fdr: Benjamini-Hochberg false discovery rate (FDR) adjusted p-value for negative selection. Controls for multiple hypothesis testing
neg|rank: rank of the gene among all genes for negative selection (1 = strongest depletion)
neg|goodsgrna: number of sgRNAs for the gene whose log-fold changes contributed to the negative-selection score after DrugZ filtering and directional consistency checks
neg|lfc: mean log₂ fold change (LFC) of sgRNAs targeting the gene, more negative values indicate stronger depletion
pos|score: gene-level p-value for positive selection (enrichment),s maller values indicate stronger evidence that disruption of this gene decreases cell fitness under the tested condition
pos|p-value:  one-sided p-value testing whether sgRNAs targeting the gene are significantly enriched relative to the background distribution
pos|fdr: Benjamini-Hochberg false discovery rate (FDR) adjusted p-value for positive selection
pos|rank: rank of the gene among all genes for positive selection (1 = strongest enrichment)
pos|goodsgrna: number of sgRNAs contributing to the positive-selection score after DrugZ filtering and directional consistency checks
pos|lfc: mean log₂ fold change (LFC) of sgRNAs targeting the gene. More positive values indicate stronger enrichment
```

=============================================================================
#### Second-round MusCK screens identify Cop1 as a regulator of TNBC progression
=============================================================================

In [11]:
# =============================================================================
# Parse in published supplemental tables
# =============================================================================
# Experiment 2 - In Vivo
MusCK2_S4_invivo = pd.read_csv(data_dir / "Table_S4_In_vivo_MusCK2_0_library_screening_on_cancer_cells.txt", sep = "\t", header = 0)
MusCK2_invivo = parse_mageck_results(MusCK2_S4_invivo, header = True)

In [12]:
# =============================================================================
# In Vivo Data - BALB/c VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK2_invivo['BALB/c VS. BALB/c Foxn1nu/nu'].describe(include = 'all').iloc[np.arange(0,4),:]

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83,83,83,83,83,83,83,83,83,83,83,83,83,83
unique,83,5,83,81,17,83,9,83,80,80,9,83,6,83
top,RFWD2,8,1.46E-12,5.00E-06,0.999905,1,1,-2.7634,1,1,0.999995,83,0,-2.7634
freq,1,78,1,3,65,1,20,1,4,4,52,1,50,1


In [13]:
# =============================================================================
# In Vivo Data - BALB/c +OVA VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK2_invivo['BALB/c +OVA VS. BALB/c Foxn1nu/nu'].describe(include = 'all').iloc[np.arange(0,4),:]

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83,83,83,83,83,83,83,83,83,83,83,83,83,83
unique,83,5,83,81,16,83,9,83,83,83,13,83,5,83
top,RFWD2,8,8.26E-14,5.00E-06,0.999585,1,0,-3.3159,1,1,0.999995,83,0,-3.3159
freq,1,78,1,3,63,1,23,1,1,1,60,1,52,1


In [14]:
# =============================================================================
# In Vivo Data - BALB/c +OVA+anti-PD-1 antibody VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK2_invivo['BALB/c +OVA+anti-PD-1 antibody VS. BALB/c Foxn1nu/nu'].describe(include = 'all').iloc[np.arange(0,4),:]

,id,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83,83,83,83,83,83,83,83,83,83,83,83,83,83
unique,83,5,83,83,20,83,10,83,82,82,7,83,8,83
top,RFWD2,8,2.01E-14,5.00E-06,0.997178,1,1,-3.6254,1,1,0.999995,83,0,-3.6254
freq,1,78,1,1,59,1,20,1,2,2,71,1,34,1


=============================================================================
#### Gene Annotation
=============================================================================